# Arrays and memory

Sections 13.1 and 13.4 of the notes.

An array is a block of memory plus a shape, a data type and strides. Everything
here follows from that: which operations move data, which only change the way
the same memory is read, and why the order of the entries decides the speed of
a loop.

## 1. Shape, data type, strides

In [ ]:
import numpy as np

x = np.arange(12, dtype=np.int64)
X = x.reshape(3, 4)

print("shape   ", x.shape)
print("dtype   ", x.dtype)
print("nbytes  ", x.nbytes)  # 12 entries x 8 bytes
print("strides ", x.strides)  # bytes to the next row, to the next column
print()
print("shape   ", X.shape)
print("dtype   ", X.dtype)
print("nbytes  ", X.nbytes)  # 12 entries x 8 bytes
print("strides ", X.strides)  # bytes to the next row, to the next column
print("X shares memory with x:", np.shares_memory(X, x))

The strides say how many bytes to jump along each axis: $32$ bytes to the next
row, $8$ bytes to the next entry of the same row. Section 3 explains this order.

## 2. Views and copies

Slicing with a range gives a **view**: a new shape and new strides on the same
memory. Indexing with a list of positions or with a boolean mask gives a
**copy**, because the selected entries are in general not equally spaced.

In [ ]:
x = np.arange(10)

view = x[2:5]
copy_fancy = x[[2, 3, 4]]
copy_mask = x[(x >= 2) & (x <= 4)]

for name, a in [("x[2:5]", view), ("x[[2,3,4]]", copy_fancy), ("x[mask]", copy_mask)]:
    print(f"{name:12s} shares memory with x: {np.shares_memory(a, x)}")

Whether a selection is a view or a copy decides what happens when you write
into it. A view shares the memory of the original array, so writing into the
view changes the original. A copy has memory of its own, so writing into it
leaves the original as it was.

This matters for functions. Python passes the array itself to a function, not
a copy, so a function that writes into its argument in place, for example
with `*=`, changes the caller's array. Passing a slice lets the function change
that part of the caller's array; passing the result of fancy indexing hands it
a copy, and the caller's array does not change.

In [ ]:
def scale_first_half(u):
    # multiplies the first half of u by 10, in place
    u[: u.size // 2] *= 10


u = np.arange(6.0)
scale_first_half(u)  # the array itself: u is changed
print("the array itself:", u)

w = np.arange(6.0)
scale_first_half(w[2:6])  # a view: the call changes w[2:4]
print("through a slice:", w)

v = np.arange(6.0)
scale_first_half(v[[2, 3, 4, 5]])  # the function received a copy
print("through a copy:", v)

## 3. The order of the entries in memory

NumPy stores arrays in C order by default: the entries are laid out row after
row. For a 3 x 4 array the memory holds `X[0,0], X[0,1], X[0,2], X[0,3], X[1,0], ...`,
so moving one step in memory moves one step along a row (the last index), and
the four entries of a row sit next to each other. This is what the strides
`(32, 8)` of Section 1 say.

The transpose exchanges the strides and moves no data: `X.T` reads the same
memory in another order.

In [ ]:
print("X.strides  ", X.strides)
print("X.T.strides", X.T.strides)
print("same memory:", np.shares_memory(X, X.T))
print()
print(
    "X   C-contiguous:",
    X.flags["C_CONTIGUOUS"],
    "F-contiguous:",
    X.flags["F_CONTIGUOUS"],
)
print(
    "X.T C-contiguous:",
    X.T.flags["C_CONTIGUOUS"],
    "F-contiguous:",
    X.T.flags["F_CONTIGUOUS"],
)

Why is `X.T` F-contiguous? Its memory is still `0, 1, 2, ..., 11`, eight bytes
apart. In `X.T`, which is 4 x 3, a step down a column (the first index) moves
8 bytes, to the next entry in memory, and a step along a row moves 32 bytes.
So the columns of `X.T`, which are the rows of `X`, are the contiguous pieces:

```
memory:  0  1  2  3 | 4  5  6  7 | 8  9 10 11
X.T  =  [[0, 4,  8],      column 0 of X.T = 0 1 2 3, next to each other
         [1, 5,  9],      column 1 of X.T = 4 5 6 7
         [2, 6, 10],      column 2 of X.T = 8 9 10 11
         [3, 7, 11]]
```

An array whose columns are contiguous is in F order (column order, named after
Fortran). `X.T` is not C-contiguous, because its rows, such as `0, 4, 8`, are
32 bytes apart in memory.

Reductions over an axis are fast in both orders, because NumPy chooses the loop
order internally. The difference appears when *we* write the loop, since a
column slice of a C-ordered array jumps through memory.

In [ ]:
n = 4000
C = np.random.default_rng(0).standard_normal((n, n))  # C order, 128 MB
F = np.asfortranarray(C)  # same values, column order

print("C.sum(axis=1), rows")
%timeit -r 3 C.sum(axis=1)
print("C.sum(axis=0), columns")
%timeit -r 3 C.sum(axis=0)

In [ ]:
def sum_rows(A):
    return sum(A[i].sum() for i in range(A.shape[0]))


def sum_cols(A):
    return sum(A[:, j].sum() for j in range(A.shape[1]))


print("our loop over rows of a C-ordered array (contiguous slices)")
%timeit -r 3 sum_rows(C)
print("our loop over columns of a C-ordered array (strided slices)")
%timeit -r 3 sum_cols(C)
print("our loop over columns of an F-ordered array (contiguous again)")
%timeit -r 3 sum_cols(F)

## 4. Strides by hand

`as_strided` builds an array from any shape and strides. It checks nothing, so
a wrong stride reads memory outside the array. For sliding windows the safe
version is `sliding_window_view`, which computes the strides and returns a
read-only view.

In [ ]:
from numpy.lib.stride_tricks import as_strided, sliding_window_view

x = np.arange(10)

print("strides of x:", x.strides)
print()

W = sliding_window_view(x, 4)  # windows of length 4, no data copied
print(W)
print("shares memory with x:", np.shares_memory(W, x))
print("writeable:", W.flags["WRITEABLE"])

In [ ]:
# the same by hand: each row starts one entry later than the previous one
s = x.strides[0]
W2 = as_strided(x, shape=(x.size - 3, 4), strides=(s, s), writeable=False)
print("equal to sliding_window_view:", np.array_equal(W, W2))

# a moving average over the windows costs no extra memory
print("moving average:", W.mean(axis=1))

## 5. Selecting entries

Slices pick entries at equal spacing and give views (Section 2). To pick
entries that are not equally spaced, index with a **boolean mask**, an array of
`True` and `False` of the same shape, which keeps the entries marked `True`, or
with a **list of positions**. Both give copies, since the chosen entries cannot
be described by strides on the original memory.

A mask on a 2-D array that has one entry per row, such as `X.sum(axis=1) > 10`,
selects whole rows. To select rows and columns at the same time, `np.ix_` turns
a row selection and a column selection into the rectangle where they cross.

In [ ]:
X = np.arange(12).reshape(3, 4)
print("X:\n", X)
print()

mask = X % 2 == 0
print("mask for even entries:\n", mask)
print("even entries:      ", X[mask])
print("share memory: ", np.shares_memory(X[mask], X))
print()

rows = X.sum(axis=1) > 10
print("mask for rows with sum > 10:", rows)
print("rows with sum > 10:\n", X[rows])  # the mask on the first axis
print()

cols = X.sum(axis=0) > 15
print("mask for columns with sum > 15:", cols)
print("columns with sum > 15:\n", X[:, cols])  # the mask on the second axis
print()

print("rows and columns at once:\n", X[np.ix_(rows, cols)])
print(
    "equal to selecting one after the other:",
    np.array_equal(X[np.ix_(rows, cols)], X[rows][:, cols]),
)